### IMPORTACION DE LIBRERIA

In [ ]:
import pandas as pd

### CARGA DE DATASET
Se carga el Dataset desde el Repo del Team. Se visualizan las primeras 4 filas y se verifica si todo se cargo bien

In [ ]:
# Carga del Dataset desde Git
df = pd.read_csv("https://raw.githubusercontent.com/Tec-IA-Proyectos-26/ObraSync_Datos_Alfa/refs/heads/main/DATOS/DATA_BASE.csv")

# Primeras 4 filas
df.head(4)


,Descripcion,Marca,Compra Total,Ingreso Total,Pendiente,Estado,Fecha,Año
0,ESPATULA PINTOR LAMINADA CABO PLASTICO - 50 MM,BIASSONI,36,NaN,36.0,INCOMPLETO,F. 29-1-2026,2026
1,AEROSOL SMART PAINT AA BLANCO BRILLANTE 350ML...,DOBLE A - AE,72,72.0,0.0,FULL,F. 29-1-2026,2026
2,PANTALON CARGO DEL NORTE MUJER - PAMPERO VERDE...,PAMPERO,2,NaN,2.0,INCOMPLETO,F. 29-1-2026,2026
3,REGULADOR GLP C/1 MAN. (PROP. Y OTROS),FERROLAN,15,15.0,0.0,FULL,F. 29-1-2026,2026


### --- LIMPIEZA ---
Tenemos dos secciones de limpieza:

*   Limpieza general = Problema tecnico (tipos de datos incorrectos)
*   Limpieza de strings = Problema visual (inconsistencias de formato)



In [ ]:
# LIMPIEZA GENERAL

# Limpiar nombres de columnas (quitar espacios invisibles y normalizar)
df.columns = df.columns.str.strip()

# Limpiar el prefijo "F. " y convertir a fecha real
# Convertir a string antes de hacer replace, replace se usa para eliminar el prefijo "F. " que a veces aparece en las fechas
df['Fecha'] = df['Fecha'].astype(str).str.replace('F. ', '', regex=False)
df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True, format='mixed')

# Convertir columnas numericas y manejar valores vacios
cols_numericas = ['Compra Total', 'Ingreso Total', 'Pendiente']
for col in cols_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Ver muestra de los datos ya limpios
print("Limpieza OK")
print(df.info())


Limpieza OK
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 635 entries, 0 to 634
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Descripcion    629 non-null    object        
 1   Marca          596 non-null    object        
 2   Compra Total   635 non-null    float64       
 3   Ingreso Total  635 non-null    float64       
 4   Pendiente      635 non-null    float64       
 5   Estado         543 non-null    object        
 6   Fecha          635 non-null    datetime64[ns]
 7   Año            635 non-null    int64         
dtypes: datetime64[ns](1), float64(3), int64(1), object(3)
memory usage: 39.8+ KB
None


In [ ]:
# LIMPIEZA DE STRINGS

# Limpiar y estandarizar texto en columnas principales
df['Descripcion'] = df['Descripcion'].str.strip().str.capitalize()
df['Marca'] = df['Marca'].str.strip().str.upper()

# Estandarizar la columna Estado
df['Estado'] = df['Estado'].str.strip().str.upper()

# Verificar resultado
print("Limpieza de strings completada")
print(df[['Descripcion', 'Marca', 'Estado']].head())

Limpieza de strings completada
                                         Descripcion         Marca      Estado
0     Espatula pintor laminada cabo plastico - 50 mm      BIASSONI  INCOMPLETO
1  Aerosol smart paint aa blanco brillante  350ml...  DOBLE A - AE        FULL
2  Pantalon cargo del norte mujer - pampero verde...       PAMPERO  INCOMPLETO
3             Regulador glp c/1 man. (prop. y otros)      FERROLAN        FULL
4              Aerosol uso general negro mate 250 gr       TEKBOND  INCOMPLETO


### --- ESTANDARIZACION ---

In [ ]:
# Unificar marcas similares
mapeo_marcas = {
    'DOBLE A - AE': 'DOBLE A',
    'DOBLE A - LIJ': 'DOBLE A',
    'LINCOLN EQU': 'LINCOLN'
}
df['Marca_Unificada'] = df['Marca'].replace(mapeo_marcas)

# Estandarizacion numerica (media 0, desviacion 1)
# Se dejo esto para mas adelante poder armar graficos
df['Ingreso_Std'] = (df['Ingreso Total'] - df['Ingreso Total'].mean()) / df['Ingreso Total'].std()

In [ ]:
# RESUMEN
print(f"Marcas unicas: {df['Marca_Unificada'].nunique()}")
print("\nTop 5 Marcas con mas movimientos:")
print(df['Marca_Unificada'].value_counts().head())

Marcas unicas: 92

Top 5 Marcas con mas movimientos:
Marca_Unificada
SINA          37
FERRETERIA    36
DOBLE A       31
BIASSONI      30
FERROLAN      29
Name: count, dtype: int64


### --- DATASET FINAL ---

In [ ]:
# Guardamos el resultado final
df.to_csv('/content/DATA_BASE_LISTO.csv', index=False)